# Introduction to ArcPy

**ArcPy** is a comprehensive Python site package that provides a useful and productive way to perform geographic data analysis, data conversion, data management, and map automation with Python. It is the primary mechanism for interacting with the ArcGIS platform programmatically, allowing users to leverage the full power of ArcGIS Pro or ArcGIS Desktop within a Python environment.

## Key Capabilities

ArcPy provides access to geoprocessing tools as well as additional functions, classes, and modules that allow you to create simple or complex workflows.

*   **Geoprocessing:** You can run any standard tool available in the ArcGIS Toolbox (e.g., *Buffer*, *Clip*, *Spatial Join*) directly from a script.
*   **Data Access (`arcpy.da`):** This module allows for efficient reading, writing, and editing of data using cursors (e.g., `SearchCursor`, `UpdateCursor`), offering fine-grained control over attribute and geometry data.
*   **Mapping (`arcpy.mp`):** Specifically designed for ArcGIS Pro, this module allows you to automate map production, manipulate projects (`.aprx`), manage layouts, and update layer properties programmatically.
*   **Spatial Extensions:** ArcPy includes specialized modules for extensions, such as `arcpy.sa` for Spatial Analyst (raster operations) and `arcpy.na` for Network Analyst.

## Why use ArcPy?

In the context of spatial programming, ArcPy allows you to:
1.  **Automate** repetitive tasks that would be tedious in the GUI.
2.  **Document** your analysis workflows through code, making them reproducible.
3.  **Extend** the functionality of ArcGIS by creating custom script tools.

## Documentation

For more detailed information and the complete API reference, please refer to the official Esri documentation:

**What is ArcPy? - ArcGIS Pro Documentation**


---
## 1. Environment & Setup

Before writing a single line of geoprocessing code, you need to prepare the **environment** that arcpy will operate in. Think of this like setting up a workshop before you start a project — you need to know where your tools and materials are kept, and you want to agree on a few ground rules before work begins.

There are three things to handle at the start of every arcpy script:

1. **Import and verify arcpy** — confirm the package is available and a valid license is active.
2. **Set a workspace** — tell arcpy where to look for and save data by default.
3. **Configure environment settings** — establish global preferences (like whether to overwrite existing files) that all tools will respect.

Each of these is explained in detail in the subsections below.

In [8]:
# Always import arcpy first.
# This will raise an ImportError if ArcGIS Pro is not installed,
# or a RuntimeError if no license is available.
import arcpy

print("arcpy imported successfully!")
print(f"arcpy version: {arcpy.GetInstallInfo()['Version']}")

arcpy imported successfully!
arcpy version: 3.6.2


### 1.1 Checking the License

`arcpy.CheckProduct()` returns the license status for a given product level:
- `'ArcInfo'` → Advanced
- `'ArcEditor'` → Standard
- `'ArcView'` → Basic

Some geoprocessing tools require an Advanced license or a specific extension.

In [9]:
# Check available license level
license_status = arcpy.CheckProduct('ArcInfo')  # Advanced
print(f"Advanced license status: {license_status}")

# Check if a specific extension is available (e.g., Spatial Analyst)
ext_status = arcpy.CheckExtension('Spatial')
print(f"Spatial Analyst extension status: {ext_status}")

# To check out an extension so tools can use it:
# arcpy.CheckOutExtension('Spatial')
# ... run tools that require it ...
# arcpy.CheckInExtension('Spatial')  # always return it when done

Advanced license status: AlreadyInitialized
Spatial Analyst extension status: Available


### 1.2 Setting the Workspace

#### What is a workspace?

A **workspace** is the default location on your computer — either a folder or a geodatabase — where arcpy will look for input data and save output data when you don't provide a full file path.

Without a workspace set, you would need to write out the complete path every single time you reference a dataset:

```python
# Without a workspace — repetitive and error-prone
arcpy.analysis.Buffer(
    r"C:\Users\georg\arcpy_demo\demo.gdb\fire_stations",
    r"C:\Users\georg\arcpy_demo\demo.gdb\fire_stations_buffer",
    "500 Meters"
)
```

With a workspace set, arcpy automatically prepends that path for you, so you only need to use the dataset name:

```python
# With a workspace set — clean and readable
arcpy.env.workspace = r"C:\Users\georg\arcpy_demo\demo.gdb"
arcpy.analysis.Buffer("fire_stations", "fire_stations_buffer", "500 Meters")
```

#### Types of workspaces

| Workspace type | Example path | Common use |
|---|---|---|
| File geodatabase (`.gdb`) | `C:\data\project.gdb` | Most common — stores vector, raster, and table data |
| Folder | `C:\data\shapefiles` | Working with shapefiles or other file-based formats |
| Enterprise geodatabase | Database connection file (`.sde`) | Multi-user editing in an organization |

> **Tip:** File geodatabases (`.gdb`) are preferred over folders for most workflows because they support longer field names, larger datasets, and relationship classes.
>
> **ArcGIS Pro documentation:** [Setting the workspace](https://pro.arcgis.com/en/pro-app/latest/arcpy/geoprocessing_and_python/setting-environments.htm)

> **Note:** Update the path below to a folder or file geodatabase that exists on your machine.

In [14]:
import os

#This line below only works if the notebook is run inside of ArcGIS Pro and gets the path to the current arcgis project.
# since thd data I want to use is in he current project, can set my workspace to use that same directory.
# the OS module will get that for me
aprx_path = os.path.dirname(arcpy.mp.ArcGISProject("CURRENT").filePath)
print(aprx_path) 
# Set the workspace — change this path to match your data location
arcpy.env.workspace = f"{aprx_path}\\LB.gdb"

# Verify the workspace is set correctly
print(f"Workspace: {arcpy.env.workspace}")
print(f"Workspace exists: {os.path.exists(arcpy.env.workspace)}")

C:\courses\spa\spaWeek05
Workspace: C:\courses\spa\spaWeek05\LB.gdb
Workspace exists: True


### 1.3 Common Environment Settings

#### What are environment settings?

**Environment settings** are a set of global preferences that control how geoprocessing tools behave. Every tool you run in arcpy automatically reads these settings — you set them once and they apply to everything that runs afterward, unless a tool explicitly overrides them.

You can think of environment settings like the preferences panel in an application: you configure them at the start and then forget about them, knowing that all the tools you use will honor your choices.

In arcpy, all environment settings are accessed through the `arcpy.env` object:

```python
arcpy.env.<setting_name> = value
```

#### Why do environment settings matter?

Without environment settings, every tool would need to be told individually:
- "Where should I save the output?"
- "Can I overwrite an existing file with that name?"
- "What coordinate system should the output be in?"

Environment settings answer those questions once, for all tools.

#### The most important settings to know

| Setting | What it controls | Example value |
|---|---|---|
| `arcpy.env.workspace` | Default input/output location | `r"C:\data\project.gdb"` |
| `arcpy.env.overwriteOutput` | Whether existing outputs can be replaced | `True` or `False` |
| `arcpy.env.outputCoordinateSystem` | Coordinate system applied to new datasets | `arcpy.SpatialReference(4326)` |
| `arcpy.env.extent` | Spatial bounding box — tools only process data within it | `"MAXOF"` or a specific extent |
| `arcpy.env.cellSize` | Default cell size for raster outputs | `30` (in map units) |

#### `overwriteOutput` — the setting you'll use most often

By default, `arcpy.env.overwriteOutput` is `False`. This means if you try to save a new dataset to a path where one already exists, arcpy will raise an error and stop. During development and testing, you'll typically want to set this to `True` so you can re-run your script without manually deleting old outputs first.

```python
arcpy.env.overwriteOutput = True   # safe for development
arcpy.env.overwriteOutput = False  # safer for production (avoids accidental overwrites)
```

> **ArcGIS Pro documentation:** [An overview of geoprocessing environment settings](https://pro.arcgis.com/en/pro-app/latest/tool-reference/environment-settings/an-overview-of-geoprocessing-environment-settings.htm)

In [15]:
# Allow tools to overwrite existing outputs (very useful during development)
arcpy.env.overwriteOutput = True

# Print a summary of key environment settings
print(f"Workspace:         {arcpy.env.workspace}")
print(f"Overwrite output:  {arcpy.env.overwriteOutput}")
print(f"Output coord sys:  {arcpy.env.outputCoordinateSystem}")
print(f"Extent:            {arcpy.env.extent}")

Workspace:         C:\courses\spa\spaWeek05\LB.gdb
Overwrite output:  True
Output coord sys:  None
Extent:            None


---
## 2. Describing Data

Before processing data, it's good practice to **inspect** it — understand its geometry type, coordinate system, fields, and more. arcpy provides several tools for this.

### 2.1 Listing Datasets

arcpy's `List*` functions return a list of dataset names in the current workspace.

| Function | Returns |
|---|---|
| `arcpy.ListFeatureClasses()` | Feature classes (points, lines, polygons) |
| `arcpy.ListRasters()` | Raster datasets |
| `arcpy.ListTables()` | Standalone tables |
| `arcpy.ListDatasets()` | Feature datasets (containers inside a GDB) |
| `arcpy.ListFields(fc)` | Fields in a specific feature class |

In [22]:
# List all feature classes in the workspace
feature_classes = arcpy.ListFeatureClasses()

if feature_classes:
    print(f"Found {len(feature_classes)} feature class(es):")
    for fc in feature_classes:
        print(f"  - {fc}")
else:
    print("No feature classes found in the current workspace.")

Found 12 feature class(es):
  - FloodZone100year_LagunaBeachCity
  - FireStations
  - LB_Streets
  - Trails_on_Public_Property
  - Trails_on_Private_Property
  - Parcels_LagunaBeachCity
  - LB_BuildingFootprints
  - LB_Zoning
  - LB_Police_Districts
  - LB_CBlocks
  - LB_CBlockGroups
  - LB_OpenSpace


In [23]:
# You can filter by geometry type: 'Point', 'Polyline', 'Polygon', 'MultiPatch'
polygon_fcs = arcpy.ListFeatureClasses(feature_type='Polygon')
print(f"Polygon feature classes: {polygon_fcs}")

# Wildcards work too — list anything starting with 'land'
land_fcs = arcpy.ListFeatureClasses(wild_card='land*')
print(f"Feature classes starting with 'land': {land_fcs}")

Polygon feature classes: ['FloodZone100year_LagunaBeachCity', 'Parcels_LagunaBeachCity', 'LB_BuildingFootprints', 'LB_Zoning', 'LB_Police_Districts', 'LB_CBlocks', 'LB_CBlockGroups', 'LB_OpenSpace']
Feature classes starting with 'land': []


### 2.2 Using arcpy.Describe()

`arcpy.Describe()` returns a **Describe object** with properties that vary depending on the data type. It's the go-to function for inspecting a single dataset.

In [24]:
# Replace 'your_feature_class' with an actual FC name from your workspace
fc_name = "LB_Streets"

desc = arcpy.Describe(fc_name)

print(f"Name:             {desc.name}")
print(f"Data type:        {desc.dataType}")
print(f"Shape type:       {desc.shapeType}")
print(f"Spatial reference: {desc.spatialReference.name}")
print(f"Has Z values:     {desc.hasZ}")
print(f"Has M values:     {desc.hasM}")
print(f"Extent:           {desc.extent}")

Name:             LB_Streets
Data type:        FeatureClass
Shape type:       Polyline
Spatial reference: WGS_1984_Web_Mercator_Auxiliary_Sphere
Has Z values:     False
Has M values:     False
Extent:           -13115581.7768 3959933.9684 -13105895.5364 3974959.0601 NaN NaN NaN NaN


### 2.3 Listing Fields

`arcpy.ListFields()` returns a list of **Field objects**, each with properties like `name`, `type`, `length`, and `aliasName`.

In [25]:
# List all fields in a feature class
fields = arcpy.ListFields(fc_name)

print(f"{'Field Name':<25} {'Type':<15} {'Length':<10} {'Alias'}")
print("-" * 65)
for field in fields:
    print(f"{field.name:<25} {field.type:<15} {field.length:<10} {field.aliasName}")

Field Name                Type            Length     Alias
-----------------------------------------------------------------
OBJECTID                  OID             4          OBJECTID
Shape                     Geometry        0          Shape
LENGTH                    Double          8          LENGTH
STREET_                   Double          8          STREET_
STREET_ID                 Double          8          STREET_ID
PVMNTTYPE                 String          3          PVMNTTYPE
PVMNTWDTH                 Double          8          PVMNTWDTH
SCCLCLASS                 String          11         SCCLCLASS
SCROUTENO                 String          9          SCROUTENO
SCSTCLASS                 String          51         SCSTCLASS
USESTATUS                 Double          8          USESTATUS
SHLDERTYPE                String          3          SHLDERTYPE
STBASETYPE                String          3          STBASETYPE
TOSTNAME                  Double          8          TOSTNAME
TO

In [26]:
# Filter fields by type — useful when you want only numeric or string fields
numeric_types = ['SmallInteger', 'Integer', 'Single', 'Double']
numeric_fields = [f.name for f in fields if f.type in numeric_types]

print(f"Numeric fields: {numeric_fields}")

# Exclude arcpy's auto-managed fields (OID, geometry, etc.)
excluded = ['OID', 'Geometry', 'GlobalID', 'Guid']
user_fields = [f.name for f in fields if f.type not in excluded]
print(f"User-defined fields: {user_fields}")

Numeric fields: ['LENGTH', 'STREET_', 'STREET_ID', 'PVMNTWDTH', 'USESTATUS', 'TOSTNAME', 'STLIMITS', 'TOMAINTORG', 'MAPID', 'CITY_CODE', 'Shape_Length']
User-defined fields: ['LENGTH', 'STREET_', 'STREET_ID', 'PVMNTTYPE', 'PVMNTWDTH', 'SCCLCLASS', 'SCROUTENO', 'SCSTCLASS', 'USESTATUS', 'SHLDERTYPE', 'STBASETYPE', 'TOSTNAME', 'TOSTREET', 'LOCALAUTH', 'STCLASSCO', 'STCLASSST', 'STLIMITS', 'STTYPE', 'SUBBASETYP', 'TOMAINTORG', 'MAPID', 'CITY_CODE', 'CITY_STREE', 'Shape_Length']


---
## 3. Geoprocessing Tools

arcpy exposes every ArcGIS Pro geoprocessing tool as a Python function. Tools are organized into **toolboxes** accessed via dot notation:

```
arcpy.<toolbox_alias>.<ToolName>(parameters)
```

For example, the Buffer tool in the Analysis toolbox is `arcpy.analysis.Buffer()`.

> **Tip:** Every tool call returns a **Result object**. Check `result.status` (4 = succeeded) or use `arcpy.GetMessages()` to inspect tool output.

### 3.1 Buffer

**Buffer** creates polygon areas around input features at a specified distance. Common use cases: proximity analysis, impact zones, service areas.

**Signature:**
```python
arcpy.analysis.Buffer(
    in_features,
    out_feature_class,
    buffer_distance_or_field,
    line_side="FULL",        # FULL | LEFT | RIGHT | OUTSIDE_ONLY
    line_end_type="ROUND",   # ROUND | FLAT
    dissolve_option="NONE",  # NONE | ALL | LIST
    dissolve_field=None
)
```

In [27]:
# --- Buffer Example ---
# Input: a point feature class (e.g., fire stations, schools)
# Output: 500-meter buffer polygons around each point

in_fc   = "FireStations"     # replace with your feature class name
out_fc  = "FireStations_buffer_500m"
distance = "500 Meters"

result = arcpy.analysis.Buffer(
    in_features=in_fc,
    out_feature_class=out_fc,
    buffer_distance_or_field=distance,
    dissolve_option="NONE"    # keep individual buffers per feature
)

print(f"Buffer completed. Output: {result.getOutput(0)}")
print(arcpy.GetMessages())

Buffer completed. Output: C:\courses\spa\spaWeek05\LB.gdb\FireStations_buffer_500m
Start Time: Wednesday, February 25, 2026 5:57:15 PM
Succeeded at Wednesday, February 25, 2026 5:57:16 PM (Elapsed Time: 0.95 seconds)


In [28]:
# Dissolved buffer — merges all buffers into a single polygon
# Useful when you only care about the combined coverage area

out_fc_dissolved = "FireStations_buffer_dissolved"

result = arcpy.analysis.Buffer(
    in_features=in_fc,
    out_feature_class=out_fc_dissolved,
    buffer_distance_or_field=distance,
    dissolve_option="ALL"     # dissolve into one polygon
)

print(f"Dissolved buffer completed. Output: {result.getOutput(0)}")

Dissolved buffer completed. Output: C:\courses\spa\spaWeek05\LB.gdb\FireStations_buffer_dissolved


### 3.2 Clip

**Clip** extracts features from an input layer that fall within the boundary of a clip layer — like a "cookie cutter" for GIS data.

**Signature:**
```python
arcpy.analysis.Clip(
    in_features,
    clip_features,
    out_feature_class,
    cluster_tolerance=None
)
```

In [29]:
# --- Clip Example ---
# Clip a roads layer to the boundary of a study area polygon

in_features    = "LB_Streets"           # the layer to clip
clip_features  = "FloodZone100year_LagunaBeachCity"      # the boundary to clip to
out_clipped    = "roads_clipped_by_flood_zone"

result = arcpy.analysis.Clip(
    in_features=in_features,
    clip_features=clip_features,
    out_feature_class=out_clipped
)

print(f"Clip completed. Output: {result.getOutput(0)}")
print(arcpy.GetMessages())

Clip completed. Output: C:\courses\spa\spaWeek05\LB.gdb\roads_clipped_by_flood_zone
Start Time: Wednesday, February 25, 2026 6:01:35 PM
Analyzing input features...
Dissolving clip features...
Clipping input features...
Succeeded at Wednesday, February 25, 2026 6:01:36 PM (Elapsed Time: 1.01 seconds)


In [30]:
# Count features in original vs clipped to see how many were retained
original_count = int(arcpy.management.GetCount(in_features).getOutput(0))
clipped_count  = int(arcpy.management.GetCount(out_clipped).getOutput(0))

print(f"Original feature count: {original_count}")
print(f"Clipped feature count:  {clipped_count}")
print(f"Features retained:      {clipped_count / original_count:.1%}")

Original feature count: 3459
Clipped feature count:  150
Features retained:      4.3%


### 3.3 Intersect

**Intersect** computes the geometric intersection of two or more feature classes. Only features (or portions of features) that overlap in **all** inputs are kept in the output. Attributes from all inputs are joined to the result.

**Signature:**
```python
arcpy.analysis.Intersect(
    in_features,          # list of feature classes to intersect
    out_feature_class,
    join_attributes="ALL", # ALL | NO_FID | ONLY_FID
    output_type="INPUT"   # INPUT | POINT | LINE | POLYGON
)
```

> **Clip vs Intersect:** Clip is one-directional (input is cut by clip layer). Intersect is symmetric — it computes the overlap and combines attributes from both layers.

In [31]:
# --- Intersect Example ---
# Find where flood zones overlap with parcels

in_features_list = ["FloodZone100year_LagunaBeachCity", "Parcels_LagunaBeachCity"]
out_intersected  = "flood_zone_parcels"

result = arcpy.analysis.Intersect(
    in_features=in_features_list,
    out_feature_class=out_intersected,
    join_attributes="ALL",
    output_type="INPUT"   # output geometry matches input geometry type
)

print(f"Intersect completed. Output: {result.getOutput(0)}")
print(arcpy.GetMessages())

Intersect completed. Output: C:\courses\spa\spaWeek05\LB.gdb\flood_zone_parcels
Start Time: Wednesday, February 25, 2026 9:20:02 PM
Reading Features...
Cracking Features...
Assembling Features...
Succeeded at Wednesday, February 25, 2026 9:20:03 PM (Elapsed Time: 1.37 seconds)


In [32]:
# Inspect the intersected output — note it has fields from both input layers
intersect_fields = arcpy.ListFields(out_intersected)

print(f"Fields in intersected output ({len(intersect_fields)} total):")
for field in intersect_fields:
    print(f"  {field.name} ({field.type})")

Fields in intersected output (59 total):
  OBJECTID (OID)
  Shape (Geometry)
  FID_FloodZone100year_LagunaBeachCity (Integer)
  FLD_AR_ID (String)
  FLD_ZONE (String)
  FLOODWAY (String)
  SFHA_TF (String)
  STATIC_BFE (Double)
  V_DATUM (String)
  DEPTH (Double)
  LEN_UNIT (String)
  VELOCITY (Double)
  VEL_UNIT (String)
  AR_REVERT (String)
  BFE_REVERT (Double)
  DEP_REVERT (Double)
  SOURCE_CIT (String)
  VE_elev (String)
  FID_Parcels_LagunaBeachCity (Integer)
  OBJECTID_1 (Integer)
  Shape_Leng (Double)
  APN (String)
  APNC (String)
  AP (String)
  PIN (String)
  ADDRESS (String)
  OWNERLAB (String)
  AREA (Double)
  BLDGSITE (String)
  OWNERS (String)
  ADRESLAB (String)
  CSZLAB (String)
  SUBDIVIS (String)
  BLCKBLDG (String)
  LOTUNIT (String)
  LGLDSCRP (String)
  ROOMS (String)
  STORIES (String)
  NOUNITS (String)
  SALEPRCE (String)
  SALEDATE (String)
  TAXCODE (String)
  TOTLBAR (String)
  LANDVLUE (String)
  TOTIMPVU (String)
  TOTTAXVU (String)
  TOTTAXES (String)
  

---
## Excercise - Putting It All Together

# Assignment: Geoprocessing with ArcPy — Analyzing School Zones in Laguna Beach

## Overview

In this assignment, you will write a Python script using **ArcPy** to perform a series of geoprocessing operations on spatial data for the City of Laguna Beach. Your goal is to analyze the area surrounding public schools by buffering school locations, clipping road networks, intersecting zoning parcels, and dissolving features to produce summary statistics.

By the end of this assignment, you will have practiced the following geoprocessing tools programmatically:

- **Buffer**
- **Clip**
- **Intersect**
- **Dissolve**

## Data

Your workspace should contain the following feature classes:

| Feature Class | Description |
|---|---|
| `LB_Public_Schools` | Point locations of public schools in Laguna Beach |
| `LB_Streets` | Line features representing the Laguna Beach road network |
| `LB_Zoning` | Polygon features representing zoning parcels in Laguna Beach |

## Instructions

### Setup

Begin your script by importing `arcpy` and configuring your environment:

1. Set `arcpy.env.workspace` to the path of your geodatabase or folder containing the data.
2. Set `arcpy.env.overwriteOutput` to `True` so that outputs are replaced on subsequent runs.
3. Use `arcpy.ListFeatureClasses()` to list and print the available datasets. Verify that all three required feature classes are present before proceeding.

### Step 1: Buffer School Locations

Create a **1 kilometer buffer** around each school in `LB_Public_Schools`.

- **Tool:** `arcpy.analysis.Buffer()`
- **Output:** A new feature class (e.g., `schools_1km_buffer`)
- **Buffer Distance:** `"1 Kilometers"`

### Step 2: Clip Streets to School Buffers

Clip the `LB_Streets` feature class using the buffer output from Step 1. This will isolate only the road segments that fall within 1 km of a public school.

- **Tool:** `arcpy.analysis.Clip()`
- **Input Features:** `LB_Streets`
- **Clip Features:** Your buffer output from Step 1
- **Output:** A new feature class (e.g., `roads_near_schools`)

### Step 3: Intersect School Buffers with Zoning

Intersect the school buffer layer with `LB_Zoning` to determine which zoning types fall within 1 km of each school.

- **Tool:** `arcpy.analysis.Intersect()`
- **Input Features:** Your buffer output from Step 1 and `LB_Zoning`
- **Output:** A new feature class (e.g., `schools_zoning_intersect`)

### Step 4: Dissolve Clipped Roads

Dissolve the clipped roads layer from Step 2 to calculate the **total length of all roads** within 1 km of a school.

- **Tool:** `arcpy.management.Dissolve()`
- **Input:** Your clipped roads output from Step 2
- **Output:** A new feature class (e.g., `roads_dissolved`)

> **Hint:** After dissolving, the resulting geometry will contain the combined length of all road segments. You can inspect the shape length field to find the total.

### Step 5: Dissolve Zoning Intersection by Zoning Type

Dissolve the intersection output from Step 3 using the **`ZONING1`** field. This will allow you to calculate the **total area of each zoning type** within 1 km of a school.

- **Tool:** `arcpy.management.Dissolve()`
- **Input:** Your intersection output from Step 3
- **Dissolve Field:** `"ZONING1"`
- **Output:** A new feature class (e.g., `zoning_dissolved`)

> **Hint:** After dissolving on `ZONING1`, each row in the output will represent a unique zoning type. Inspect the shape area field to find the total area per type.

## Requirements

- Your script must run without errors from start to finish.
- Include a `print()` statement after each step confirming its completion (as shown in the code outline).
- Use meaningful output feature class names.
- Comment your code to explain what each section does.

## Deliverables

Your completed Python script in the cell below.


In [ ]:
import arcpy

# --- Setup ---
arcpy.env.workspace =     # update this path
arcpy.env.overwriteOutput = True

# --- Inspect data before processing ---
fcs = arcpy.ListFeatureClasses()
print(f"Datasets available: {fcs}")

# --- Step 1: Buffer LB_Public_Schools by 1 km ---

print("Step 1 complete: school buffers created")


# --- Step 2: Clip LB_Streets to the buffered school zone ---

print("Step 2 complete: roads clipped to school zones")


# --- Step 3: Intersect schools_1km_buffer with zoning parcel layer LB_Zoning ---

print("Step 3 complete: schools_1km_buffer-zoning intersection created")


# --- Step 4: Dissolve roads_near_schools to find total length of all roads within 1kn of a school ---

print("Step 4 complete: roads_near_schools dissolve complete ")


# --- Step 5: Dissolve schools_zoning_intersect using the Zoning1 Field to find total area of each zoning type within 1kn of a school ---

print("Step 5 complete: schools_zoning_intersect dissolve complete")

print("\nWorkflow finished!")

---
## Summary

| Topic | Key Functions |
|---|---|
| Environment setup | `arcpy.env.workspace`, `arcpy.env.overwriteOutput`, `arcpy.CheckExtension()` |
| Listing datasets | `arcpy.ListFeatureClasses()`, `arcpy.ListFields()`, `arcpy.ListRasters()` |
| Describing data | `arcpy.Describe()` → `.shapeType`, `.spatialReference`, `.extent` |
| Buffer | `arcpy.analysis.Buffer()` — proximity zones around features |
| Clip | `arcpy.analysis.Clip()` — cookie-cutter extraction |
| Intersect | `arcpy.analysis.Intersect()` — overlap with combined attributes |
